# LA Studio TTS — VieNeu-TTS v3 Turbo

This notebook loads exactly `vieneu-tts-v3-turbo` (`pnnbao-ump/VieNeu-TTS-v3-Turbo`) on CUDA.
It does not use API Gateway and refuses every other model ID.

1. Choose **Runtime → Change runtime type → GPU**.
2. Run all cells.
3. Copy the printed URL and token into LA Studio's TTS panel.


In [ ]:
!nvidia-smi
%pip install -q "torch==2.8.0" "torchaudio==2.8.0" --index-url https://download.pytorch.org/whl/cu128
%pip install -q --upgrade --force-reinstall --no-deps "torchvision==0.23.0" --index-url https://download.pytorch.org/whl/cu128
%pip install -q "transformers==4.57.6" "git+https://github.com/pnnbao97/VieNeu-TTS.git@f56ce97ffb37" "soundfile==0.13.1" "fastapi==0.115.12" "uvicorn==0.34.3"


# Colab can retain an older torchvision after torch is upgraded. Transformers
# then masks the binary mismatch as a missing PreTrainedModel/Qwen3 class.
import importlib.metadata as package_metadata
import traceback

import torch
import torchvision

print("PyTorch stack:", torch.__version__, torchvision.__version__)
assert torch.cuda.is_available(), "CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU."
assert package_metadata.version("torchvision").split("+")[0] == "0.23.0", "VieNeu requires torchvision 0.23.0 with torch 2.8.0."
try:
    from transformers import PreTrainedModel
    from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM
except Exception as error:
    traceback.print_exc()
    raise RuntimeError(
        "The Colab PyTorch/Transformers stack is not importable for VieNeu. "
        "Restart the runtime, rerun this install cell, then run all cells again."
    ) from error
print("Transformers imports verified for VieNeu:", PreTrainedModel.__name__, Qwen3ForCausalLM.__name__)


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_tts_worker.py')
WORKER.write_text('import io\nimport os\nimport threading\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom fastapi import FastAPI, Header, HTTPException\nfrom fastapi.responses import Response\nfrom pydantic import BaseModel, Field\n\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.")\n\nTOKEN = os.environ["LA_STUDIO_COLAB_TTS_TOKEN"]\nMAX_INPUT_CHARS = 4000\nMAX_OUTPUT_SECONDS = 300\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\n\nclass SpeechRequest(BaseModel):\n    model: str = Field(min_length=1, max_length=120)\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\n    voice: str = Field(default="auto", max_length=160)\n    language: str = Field(default="auto", max_length=40)\n    speed: float = Field(default=1.0, ge=0.25, le=4.0)\n    response_format: str = "wav"\n    settings: dict = Field(default_factory=dict)\n\ndef authorize(authorization: str | None) -> None:\n    if authorization != "Bearer " + TOKEN:\n        raise HTTPException(status_code=401, detail="invalid worker token")\n\ndef wav_response(samples, sample_rate: int):\n    audio = np.asarray(samples, dtype=np.float32).reshape(-1)\n    if audio.size == 0:\n        raise RuntimeError("the selected model returned no audio")\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\n        raise HTTPException(status_code=413, detail="generated audio exceeds the five minute output limit")\n    if not np.isfinite(audio).all():\n        raise RuntimeError("the selected model returned non-finite audio")\n    peak = float(np.max(np.abs(audio)))\n    if peak > 1.2:\n        audio = audio / peak\n    output = io.BytesIO()\n    sf.write(output, audio, int(sample_rate), format="WAV", subtype="PCM_16")\n    return Response(output.getvalue(), media_type="audio/wav", headers={"Cache-Control": "no-store"})\n\nfrom vieneu import Vieneu\n\nMODEL_ID = "vieneu-tts-v3-turbo"\nMODEL_NAME = "VieNeu-TTS v3 Turbo"\nUPSTREAM_MODEL = "pnnbao-ump/VieNeu-TTS-v3-Turbo"\nSUPPORTED_LANGUAGES = ["vi", "en"]\nMODEL = Vieneu(mode="v3turbo", device="cuda", backend="pytorch", backbone_repo=UPSTREAM_MODEL)\nif getattr(MODEL, "backend", "") != "pytorch":\n    raise RuntimeError("VieNeu v3 Turbo did not activate the PyTorch CUDA backend")\nVOICE_ROWS = MODEL.list_preset_voices()\nSUPPORTED_VOICES = [str(row[1]) for row in VOICE_ROWS] if VOICE_ROWS else ["auto"]\n\ndef synthesize_exact_model(request: SpeechRequest):\n    voice = request.voice.strip()\n    kwargs = {"text": request.input}\n    if voice and voice.lower() != "auto":\n        kwargs["voice"] = voice\n    style = str(request.settings.get("style", "")).strip()\n    if style:\n        kwargs["style"] = style\n    audio = MODEL.infer(**kwargs)\n    return audio, 48000\n\napp = FastAPI(title=f"LA Studio TTS — {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "gpu": torch.cuda.get_device_name(0),\n        "model": MODEL_ID,\n        "upstream_model": UPSTREAM_MODEL,\n        "cpu_fallback": False,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "tts",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "upstream_model": UPSTREAM_MODEL,\n                "languages": SUPPORTED_LANGUAGES,\n                "voices": SUPPORTED_VOICES,\n                "formats": ["wav"],\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n@app.post("/v1/audio/speech")\ndef speech(request: SpeechRequest, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    if request.model.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{request.model}\'. Open the notebook for the selected model.",\n        )\n    if request.response_format.strip().lower() != "wav":\n        raise HTTPException(status_code=422, detail="this worker returns WAV audio only")\n    if not REQUEST_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="the Colab TTS worker is busy; retry shortly")\n    try:\n        samples, sample_rate = synthesize_exact_model(request)\n        return wav_response(samples, sample_rate)\n    except HTTPException:\n        raise\n    except Exception as error:\n        raise HTTPException(\n            status_code=503,\n            detail=f"{MODEL_NAME} synthesis failed: {type(error).__name__}: {str(error)[:240]}",\n        ) from error\n    finally:\n        REQUEST_SLOTS.release()\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
MODEL_ID = 'vieneu-tts-v3-turbo'

import json, os, re, secrets, subprocess, sys, time, urllib.error, urllib.request
from pathlib import Path

TOKEN = secrets.token_urlsafe(32)
STARTUP_TIMEOUT_SECONDS = 20 * 60
WORKER_LOG = Path("/content/la_studio_tts_worker.log")
env = os.environ.copy()
env["LA_STUDIO_COLAB_TTS_TOKEN"] = TOKEN
env["PYTHONUNBUFFERED"] = "1"

def worker_log_tail() -> str:
    try:
        return WORKER_LOG.read_text(encoding="utf-8", errors="replace")[-12000:]
    except FileNotFoundError:
        return "(worker log was not created)"

def fail_startup(message: str) -> None:
    if worker.poll() is None:
        worker.terminate()
        try:
            worker.wait(timeout=10)
        except subprocess.TimeoutExpired:
            worker.kill()
    raise RuntimeError(
        message + "\\n\\n---- LA Studio TTS worker log (last 12,000 characters) ----\\n" + worker_log_tail()
    )

with WORKER_LOG.open("w", encoding="utf-8", buffering=1) as worker_output:
    worker = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "la_studio_tts_worker:app", "--host", "127.0.0.1", "--port", "3921"],
        cwd="/content",
        env=env,
        stdout=worker_output,
        stderr=subprocess.STDOUT,
    )
    print("Starting exact CUDA TTS worker; initial model download can take several minutes.")
    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS
    next_report = time.monotonic()
    last_error = "worker has not answered /health yet"
    while time.monotonic() < deadline:
        exit_code = worker.poll()
        if exit_code is not None:
            fail_startup(f"The exact-model TTS worker exited before becoming ready (exit code {exit_code}).")
        try:
            check = urllib.request.Request(
                "http://127.0.0.1:3921/health",
                headers={"Authorization": "Bearer " + TOKEN},
            )
            with urllib.request.urlopen(check, timeout=10) as response:
                health = json.loads(response.read().decode("utf-8"))
            if (response.status == 200
                    and health.get("ready") is True
                    and health.get("device") == "cuda"
                    and health.get("model") == MODEL_ID
                    and health.get("cpu_fallback") is False):
                print("Exact CUDA TTS worker is ready:", health)
                break
            last_error = "unexpected /health response: " + json.dumps(health, ensure_ascii=False)
        except urllib.error.HTTPError as error:
            last_error = f"/health returned HTTP {error.code}: " + error.read().decode("utf-8", errors="replace")[:1000]
        except Exception as error:
            last_error = f"/health is not ready: {type(error).__name__}: {error}"
        if time.monotonic() >= next_report:
            print("Waiting for the exact CUDA TTS model…", last_error)
            next_report = time.monotonic() + 30
        time.sleep(2)
    else:
        fail_startup(
            f"The exact-model TTS worker did not become CUDA-ready within {STARTUP_TIMEOUT_SECONDS // 60} minutes. "
            f"Last health-check result: {last_error}"
        )

subprocess.run(
    ["bash", "-lc", "wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i /content/cloudflared.deb"],
    check=True,
)
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:3921", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
public_url = None
for _ in range(120):
    line = tunnel.stdout.readline()
    print(line, end="")
    match = re.search(r"https://[^\s]+trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    worker.terminate()
    tunnel.terminate()
    raise RuntimeError("Cloudflare tunnel URL was not found")

print("\nLA_STUDIO_COLAB_TTS_URL=" + public_url)
print("LA_STUDIO_COLAB_TTS_TOKEN=" + TOKEN)
print("LA_STUDIO_COLAB_TTS_MODEL=" + MODEL_ID)
print("DEVICE=cuda; CPU_FALLBACK=false")
